In [0]:
%sql
use catalog adb_rtp;
insert into gold.reporting_fact_daily_pricing_gold
select datedim.Date_id, statedim.STATE_ID,marketdim.market_id, productdim.product_id,varietydim.variety_id, silverfact.ROW_ID, silverfact.ARRIVAL_IN_TONNES, silverfact.MAXIMUM_PRICE, silverfact.MINIMUM_PRICE, silverfact.MODAL_PRICE, current_timestamp(), current_timestamp()
from silver.daily_pricing_silver silverfact
left outer join gold.reporting_dim_date_gold datedim
on silverfact.DATE_OF_PRICING = datedim.calendar_date
left outer join gold.reporting_dim_state_gold statedim
on silverfact.STATE_NAME = statedim.STATE_NAME
left outer join gold.reporting_dim_market_gold marketdim
on silverfact.MARKET_NAME = marketdim.MARKET_NAME
left outer join gold.reporting_dim_variety_gold varietydim
on silverfact.VARIETY = varietydim.VARIETY
left outer join gold.reporting_dim_product_gold productdim
on silverfact.PRODUCT_NAME = productdim.PRODUCT_NAME
and silverfact.PRODUCTGROUP_NAME = productdim.PRODUCTGROUP_NAME
where silverfact.lakehouse_updated_date > (select NVl(max(processed_file_table_date),"2023-05-01")from processrunlogs.deltalakehouse_process_runs where process_name = 'reporting_fact_table_load' and process_status = 'complete')

In [0]:
%sql
insert into processrunlogs.deltalakehouse_process_runs(process_name, processed_table_datetime, process_status)
select 'reporting_fact_table_load', max(lakehouse_updated_date),'Completed' from silver.daily_pricing_silver